In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from crewai import LLM

llm = LLM(
model='openai/gpt-4.1-mini',
temperature=0.4,
max_tokens=1000,
frequency_penalty=0.1,
presence_penalty=0.1,
top_p=0.9
)

### 툴 설정

In [3]:
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()

In [4]:
from crewai import Agent, Task, Crew, Process

### 에이전트 정의

In [5]:
# 정보 조사 에이전트 정의
research_agent = Agent(
    role='정보 조사자',
    goal='{place} 여행에 필요한 최신 정보를 조사하여 제공합니다.',
    backstory='온라인 정보 검색에 능통한 여행 정보 전문가입니다.',
    llm=llm,
    tools=[search_tool],            #웹 검색 도구 상자
    verbose=True
)

# 일정 작성 에이전트 정의
planner_agent = Agent(
    role='여행 일정 기획자',
    goal='제공된 정보를 활용해서 완성도 높은 {place} 여행 일정을 작성합니다.',
    backstory='여행사에서 10년 경력의 전문 여행 플래너로, 다양한 국내 여행 일정을 여러 차례 기획한 경험이 있습니다.',
    llm=llm,
    verbose=True
)

### 태스크 정의

In [8]:
research_task = Task(
    description=(
        '{place} 여행을 위해 알아야할 핵심 정보를 조사하세요. \n'       # \n : 줄 바꿈
        '{place}의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요.'
    ),
    agent=research_agent,
    expected_output='한국어로 작성된 {place} 여행에 대한 요약 정보 목록'
)

planning_task = Task(
    description=(
        '위의 조사 결과를 참고하여 {place}에서 {days}일 동안 머무는 여행 일정을 작성해 주세요. \n'
        '각 날짜별로 오전/오후/저녁 계획을 세우고, 조사된 관광지와 맛집 정보를 일정에 반영하세요. \n'
    ),
    agent=planner_agent,           # 이전 조사 결과를 컨텍스트로 전달
    expected_output='한국어로 작성된 정보를 반영한 {days} 일간의 여행 일정'
)

### 크루 생성

In [10]:
# 두 에이전트를 Crew로 묶어 순차 진행
crew_multi = Crew(
    agents=[research_agent, planner_agent],
    tasks=[research_task, planning_task],
    process=Process.sequential,
    verbose=True
)

In [11]:
place = '제주'
days = 3

print(f"=== [협업 에이전트] {place} {days}일 일정 생성 시작 ===")
result_multi = await crew_multi.kickoff_async(inputs = {'place':place, 'days':days})
print(f"=== [협업 에이전트] 생성된 {place} {days}일 일정 ===")
print(result_multi)

=== [협업 에이전트] 제주 3일 일정 생성 시작 ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b232254b-6a8b-4058-8715-b55a5dbd4b27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 제주 여행을 위해 알아야할 핵심 정보를 조사하세요.                                                        │
│  제주의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요.   │
│  ID: 8b7953ce-1969-4f5f-8f43-19bf74a58864                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 정보 조사자                                                                                             │
│                                                                                                                 │
│  Task: 제주 여행을 위해 알아야할 핵심 정보를 조사하세요.                                                        │
│  제주의 인기 관광지 목록, 지역별 맛집 추천, 이동 시 유용한 교통 정보 등을 최신 자료를 기반으로 정리해 주세요.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '제주 인기 관광지 2024'}                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 여행 일정 기획자                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  [제주 3일 여행 일정]                                                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1일차: 제주시 중심 자연과 미식 탐방                                                                        │
│                                                                                                                 │
│  **오전**                                                                                                       │
│  - 한라산국립공원 방문                                                                                          │
│    - 한라산 등반 또는 산책로 코스 선택 (성판악 코스 추천, 가벼운 산책 가능)                                     │
│    - 사계절 아름다운 자연 경관 감상 및 사진 촬영                                                                │
│                                                                                                                 │
│  **점심**                                                                                                       │
│  - 제주시 우진해장국 방문                                                                                       │
│    - 제주 대표 해장국 맛집에서 든든한 한 끼                                                                     │
│                                                                                                                 │
│  **오후**                                                                                                       │
│  - 돌문화공원 탐방                                                                                              │
│    - 제주의 독특한 돌 문화와 자연을 함께 체험                                                                   │
│  - 용두암 방문                                                                                                  │
│    - 제주 바다와 용 모양 바위 감상하며 산책                                                                     │
│                                                                                                                 │
│  **저녁**                                                                                                       │
│  - 제주시 장인의집 또는 해녀촌 방문                                                                             │
│    - 신선한 회국수 또는 제주 해산물 요리 맛보기                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2일차: 서귀포 중문과 힐링 숲길                                                                             │
│                                                                                                                 │
│  **오전**                                                                                                       │
│  - 사려니숲길 산책                                                                                              │
│    - 맑은 공기와 울창한 숲길에서 힐링하며 여유로운 아침 시작  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 위의 조사 결과를 참고하여 제주에서 3일 동안 머무는 여행 일정을 작성해 주세요.                            │
│  각 날짜별로 오전/오후/저녁 계획을 세우고, 조사된 관광지와 맛집 정보를 일정에 반영하세요.                       │
│                                                                                                                 │
│  Agent: 여행 일정 기획자                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b232254b-6a8b-4058-8715-b55a5dbd4b27                                                                       │
│  Final Output: [제주 3일 여행 일정]                                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1일차: 제주시 중심 자연과 미식 탐방                                                                        │
│                                                                                                                 │
│  **오전**                                                                                                       │
│  - 한라산국립공원 방문                                                                                          │
│    - 한라산 등반 또는 산책로 코스 선택 (성판악 코스 추천, 가벼운 산책 가능)                                     │
│    - 사계절 아름다운 자연 경관 감상 및 사진 촬영                                                                │
│                                                                                                                 │
│  **점심**                                                                                                       │
│  - 제주시 우진해장국 방문                                                                                       │
│    - 제주 대표 해장국 맛집에서 든든한 한 끼                                                                     │
│                                                                                                                 │
│  **오후**                                                                                                       │
│  - 돌문화공원 탐방                                                                                              │
│    - 제주의 독특한 돌 문화와 자연을 함께 체험                                                                   │
│  - 용두암 방문                                                                                                  │
│    - 제주 바다와 용 모양 바위 감상하며 산책                                                                     │
│                                                                                                                 │
│  **저녁**                                                                                                       │
│  - 제주시 장인의집 또는 해녀촌 방문                                                                             │
│    - 신선한 회국수 또는 제주 해산물 요리 맛보기                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2일차: 서귀포 중문과 힐링 숲길                                                                             │
│                                                                                                                 │
│  **오전**                                                                                                       │
│  - 사려니숲길 산책                                                                                              │
│    - 맑은 공기와 울창한 숲길에서 힐링하며 여유로운

=== [협업 에이전트] 생성된 제주 3일 일정 ===
[제주 3일 여행 일정]

---

### 1일차: 제주시 중심 자연과 미식 탐방

**오전**  
- 한라산국립공원 방문  
  - 한라산 등반 또는 산책로 코스 선택 (성판악 코스 추천, 가벼운 산책 가능)  
  - 사계절 아름다운 자연 경관 감상 및 사진 촬영  

**점심**  
- 제주시 우진해장국 방문  
  - 제주 대표 해장국 맛집에서 든든한 한 끼  

**오후**  
- 돌문화공원 탐방  
  - 제주의 독특한 돌 문화와 자연을 함께 체험  
- 용두암 방문  
  - 제주 바다와 용 모양 바위 감상하며 산책  

**저녁**  
- 제주시 장인의집 또는 해녀촌 방문  
  - 신선한 회국수 또는 제주 해산물 요리 맛보기  

---

### 2일차: 서귀포 중문과 힐링 숲길

**오전**  
- 사려니숲길 산책  
  - 맑은 공기와 울창한 숲길에서 힐링하며 여유로운 아침 시작  

**점심**  
- 중문 엘엠엔티 또는 비스트로 낭 예약 방문  
  - 중문 지역 인기 맛집에서 고급스러운 식사 경험  

**오후**  
- 중문색달해수욕장 방문  
  - 해변 산책 및 바닷가 풍경 감상, 여름철이면 피서 즐기기  
- 아쿠아플라넷 제주 방문 (선택 사항)  
  - 다양한 해양 생물 관람 및 체험  

**저녁**  
- 중문 지역 맛집 재방문 또는 인근 도민 추천 찐 맛집 탐방 (예: 운산 식당)  
  - 제주 향토 음식과 신선한 재료 활용한 저녁 식사  

---

### 3일차: 동부 해변과 테마파크, 현지 맛집

**오전**  
- 함덕해변 산책 및 카페 방문  
  - 함덕의 아름다운 해변에서 여유로운 아침 시간 보내기  

**점심**  
- 함덕 오가네 전복 설렁탕 또는 곱들락 방문  
  - 제주 특산 전복과 해산물 요리로 점심 식사  

**오후**  
- 카멜리아 힐 방문  
  - 계절별 꽃과 아름다운 정원 산책  
- 빛의 벙커 또는 9.81 파크 방문 (시간

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯